In [2]:
import numpy as np
from usearch.index import Index, Matches
import h5py
import pandas as pd
import numpy as np

In [3]:
dataset_dict = {
    "mnist": "./data/mnist-784-euclidean.hdf5",
    "sift": "./data/sift-128-euclidean.hdf5",
}

dataset = "mnist"
sample_size = 0.4

def get_data():
    with h5py.File(dataset_dict[dataset], "r") as f:
        train = f["train"][()]
        return train

def get_test():
    with h5py.File(dataset_dict[dataset], "r") as f:
        test = f["test"][()]
        return test

def transform_data(data):
    df = pd.DataFrame(data)
    df.insert(0, "id", range(0, len(df)))  # Add unique ID column
    df["vec"] = df.apply(lambda x: x.values[1:].tolist(), axis=1)  # Exclude ID from vector
    return df[["id", "vec"]]


train_full = transform_data(get_data())
train_reduced = train_full[:int(len(train_full) * sample_size)]

test = transform_data(get_test())

dimensionality = len(train_reduced['vec'].iloc[0])

def get_ground_truth():
    """Loads ground-truth nearest neighbors from HDF5 file"""
    with h5py.File(dataset_dict[dataset], "r") as f:
        return f["neighbors"][()]  # Nearest neighbor indices


ground_truth = get_ground_truth()




In [4]:
index = Index(
    ndim=dimensionality, # Define the number of dimensions in input vectors
    metric='l2sq', # Choose 'l2sq', 'haversine' or other metric, default = 'ip'
    dtype='f32', # Quantize to 'f16' or 'i8' if needed, default = 'f32'
)



for i in range(0, len(train_reduced)):
    index.add(i, np.array(train_reduced['vec'][i]))

In [5]:
import time
import numpy as np

deleted_entries = []  # Set to store deleted entries

def compute_recall():
    total_recall = 0
    total_search_time = 0
    num_queries = 100
    k = 100

    min_time = float('inf')
    max_time = float('-inf')

    # Get all valid keys in the index
    valid_index_keys = set(np.array(index.keys))  # Assuming index.keys() gives all active keys


    for _ in range(num_queries):
        random_number = np.random.randint(0, test.shape[0])
        vec = test['vec'][random_number]

        # Measure search time
        start_time = time.time()
        retrieved_neighbors = index.search(np.array(vec), k).keys  # Get retrieved keys
        end_time = time.time()

        query_time = end_time - start_time
        total_search_time += query_time

        # Update min/max times
        min_time = min(min_time, query_time)
        max_time = max(max_time, query_time)

        # Get ground truth neighbors
        ground_truth_neighbors = ground_truth[random_number][:k]

        # **Filter out invalid points from both retrieved and ground truth sets**
        retrieved_set = set(retrieved_neighbors) & valid_index_keys
        ground_truth_set = set(ground_truth_neighbors) & valid_index_keys

        # Compute recall for this query
        intersection_size = len(retrieved_set & ground_truth_set)
        recall = intersection_size / len(ground_truth_set) if len(ground_truth_set) > 0 else 0
        total_recall += recall

    avg_recall = total_recall / num_queries
    avg_search_time = total_search_time / num_queries

    print(f"Avg search time: {avg_search_time:.6f} s, Min: {min_time:.6f} s, Max: {max_time:.6f} s")
    return avg_recall, avg_search_time, min_time, max_time

recall, search_time, min_time, max_time = compute_recall()
print(f"Recall: {recall:.4f}, Search time: {search_time:.6f} s, Min: {min_time:.6f} s, Max: {max_time:.6f} s")

Avg search time: 0.000362 s, Min: 0.000184 s, Max: 0.000697 s
Recall: 0.9959, Search time: 0.000362 s, Min: 0.000184 s, Max: 0.000697 s


In [6]:
import json
import os
import random

def delete_random_entries(iteration, filename, previous_batch_size, compact=False):
    total_entries = index.size
    num_to_delete = max(1, int(0.05 * total_entries))  # Ensure at least 1 deletion

    print(f"Total entries: {total_entries}, Deleting: {num_to_delete}")


    # Select random keys to delete
    rows_to_delete_keys = pd.DataFrame(np.array(index.keys)).sample(num_to_delete)


    # Store deleted entries to avoid re-deleting them

    # Step 3: Delete selected rows from the index
    for key in rows_to_delete_keys[0]:
        index.remove(key, compact=compact)
        deleted_entries.append(key)


    # Verify deletion
    new_total_entries = index.size
    print(f"Remaining entries after deletion: {new_total_entries}")
    print(f"Total deleted entries: {len(deleted_entries)}")

    # Step 4: Compute batch size for reinsertion (10% of remaining index)
    #batch_size = max(1, int(0.1 * (total_entries - num_to_delete)))

    # Step 5: Reinsert deleted vectors in batches
    remaining_vectors = train_full[int(len(train_full) * sample_size):]
    start_position = total_entries + num_to_delete  # Start position after deletion

    
   # batch = remaining_vectors.iloc[previous_batch_size:previous_batch_size + batch_size]  # Select batch
    batch = remaining_vectors.iloc[previous_batch_size:previous_batch_size + num_to_delete]  # Select batch
    for j, row in enumerate(batch.itertuples(index=False), start=start_position): 
        vec = np.array(row[1])
        index.add(row[0], vec)  # Adjust index based on order
    
    print(f"Inserted {len(batch)} vectors, total index size: {index.size}")

    # Step 6: Recompute recall after reinsertion
    recall_after_reinsertion, avg_search_time, min_time, max_time = compute_recall()
    print(f"Recall@100 after reinsertion: {recall_after_reinsertion:.4f}")

    # Step 7: Log results to JSON
    results = {
        'iteration': iteration,
        'min_query_time': min_time,
        'max_query_time': max_time,
        'average_query_time': avg_search_time,
        'total_entries': total_entries,
        'num_deleted': num_to_delete,
        'remaining_entries': new_total_entries,
        'recall_after_reinsertion': recall_after_reinsertion
    }

    # Save results to file
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            data = json.load(f)
    else:
        data = []

    data.append(results)

    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

    #return batch_size
    return num_to_delete


In [7]:
filename = f"hnsw_{dataset}_compact_same_size_index.json"
i = 0
previous_batch = 0
x = 0
while len(deleted_entries) + index.size < len(train_full):
    print("Iteration: ", i)
    x = delete_random_entries(i, filename, previous_batch, compact=True)
    i += 1
    previous_batch += x

Iteration:  0
Total entries: 24000, Deleting: 1200
Remaining entries after deletion: 22800
Total deleted entries: 1200
Inserted 1200 vectors, total index size: 24000
Avg search time: 0.000141 s, Min: 0.000087 s, Max: 0.000304 s
Recall@100 after reinsertion: 0.0552
Iteration:  1
Total entries: 24000, Deleting: 1200
Remaining entries after deletion: 22800
Total deleted entries: 2400
Inserted 1200 vectors, total index size: 24000
Avg search time: 0.000134 s, Min: 0.000085 s, Max: 0.000225 s
Recall@100 after reinsertion: 0.0605
Iteration:  2
Total entries: 24000, Deleting: 1200
Remaining entries after deletion: 22800
Total deleted entries: 3600
Inserted 1200 vectors, total index size: 24000
Avg search time: 0.000158 s, Min: 0.000102 s, Max: 0.000339 s
Recall@100 after reinsertion: 0.0472
Iteration:  3
Total entries: 24000, Deleting: 1200
Remaining entries after deletion: 22800
Total deleted entries: 4800
Inserted 1200 vectors, total index size: 24000
Avg search time: 0.000141 s, Min: 0.000